In [76]:
# Import packages
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import playergamelog, leaguedashptdefend
from nba_api.stats.library.parameters import SeasonAll
from itables import show
import matplotlib.pyplot as plt
import seaborn as sns
# Import modeling packages
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [104]:
from nba_api.stats.endpoints import PlayerGameLogs
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Define seasons from 2020-21 to 2024-25
seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

def fetch_player_game_logs(seasons):
    """Fetches all player game logs from the NBA API for given seasons."""
    all_game_logs = []
    
    for season in seasons:
        print(f"Fetching game logs for season: {season}...")
        
        try:
            game_logs = PlayerGameLogs(season_nullable=season, season_type_nullable="Regular Season")
            df = game_logs.get_data_frames()[0]
            all_game_logs.append(df)
            time.sleep(1)  # Rate-limit API requests
        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")

    return pd.concat(all_game_logs, ignore_index=True) if all_game_logs else pd.DataFrame()

# Fetch game logs
full_df = fetch_player_game_logs(seasons)

if full_df.empty:
    print("No data retrieved. Please check the API response.")
    exit()

# Handle missing values
full_df.fillna(0, inplace=True)

#choose player
# Input the player's name
player_name = input("Player Full Name (Ex. 'Kevin Durant'): ")

    # Convert player name to title case
player_name = player_name.title()
    
    # Retrieve the player's ID
chosen_player_id = full_df.loc[full_df['PLAYER_NAME'] == player_name, 'PLAYER_ID'].values[0]

# Compute aggregated season stats for each player
agg_columns = ['MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 
               'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PTS', 'PLUS_MINUS']

# Group by player and season
season_stats = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])[agg_columns].sum().reset_index()
season_stats['GP'] = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])['GAME_ID'].nunique().reset_index(drop=True)

# ✅ **Fix Shooting Percentages**
season_stats["FG_PCT"] = season_stats["FGM"] / season_stats["FGA"]
season_stats["FG3_PCT"] = season_stats["FG3M"] / season_stats["FG3A"]
season_stats["FT_PCT"] = season_stats["FTM"] / season_stats["FTA"]

# ✅ **Handle Division by Zero (Replace NaN with 0)**
season_stats.replace([float("inf"), -float("inf")], 0, inplace=True)
season_stats.fillna(0, inplace=True)

# Compute per-game and per-minute averages
for col in agg_columns:
    season_stats[col + "_PG"] = season_stats[col] / season_stats["GP"]
    season_stats[col + "_PM"] = season_stats[col] / season_stats["MIN"]
    
# Compute **Usage Rate (USG%)** = (FGA + 0.44 * FTA + TOV) / MIN
season_stats["USG_PCT"] = ((season_stats["FGA"] + 0.44 * season_stats["FTA"] + season_stats["TOV"]) / season_stats["MIN"]) * 100

# Compute scoring variance (PTS standard deviation per game)
scoring_variance = full_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'])['PTS'].std().reset_index()
scoring_variance.rename(columns={'PTS': 'PTS_STD'}, inplace=True)
season_stats = season_stats.merge(scoring_variance, on=['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR'], how='left')

# ✅ **Final NaN Handling**
season_stats.fillna(0, inplace=True)

# Extract season stats
player_seasons = season_stats[season_stats["PLAYER_ID"] == chosen_player_id]

# Normalize features for similarity calculation
features = ['MIN_PG', 'FGM_PG', 'FGA_PG', 'FG_PCT', 'FG3M_PG', 'FG3A_PG', 'FG3_PCT',
            'FTM_PG', 'FTA_PG', 'FT_PCT', 'OREB_PG', 'DREB_PG', 'REB_PG', 'AST_PG',
            'TOV_PG', 'STL_PG', 'BLK_PG', 'PTS_PG', 'PLUS_MINUS_PG', 'USG_PCT', 'PTS_STD']

scaler = StandardScaler()
season_stats_scaled = scaler.fit_transform(season_stats[features])
ja_morant_scaled = scaler.transform(player_seasons[features])

# Compute similarity scores using Cosine Similarity
similarity_scores = cosine_similarity(season_stats_scaled, ja_morant_scaled)

# Assign similarity scores to players
season_stats["SIMILARITY_SCORE"] = similarity_scores.max(axis=1)

# Strict filtering criteria:
strict_criteria = (
    (season_stats["USG_PCT"].between(player_seasons["USG_PCT"].mean() - 2, player_seasons["USG_PCT"].mean() + 2)) &
    (season_stats["PTS_STD"].between(player_seasons["PTS_STD"].mean() - player_seasons["PTS_STD"].std(),
                                     player_seasons["PTS_STD"].mean() + player_seasons["PTS_STD"].std()))
)

# Filter players that match Ja Morant in **Usage Rate & Scoring Variance**
similar_players = season_stats[strict_criteria].sort_values(by="SIMILARITY_SCORE", ascending=False)
similar_players


# Step 1: Extract the top 5 most similar players & their best season
top_similar_players = similar_players[1:6][['PLAYER_ID', 'PLAYER_NAME', 'SEASON_YEAR']]
print("Top 5 similar players:\n", top_similar_players)

# Step 2: Convert all_game_logs into a single DataFrame (if not already done)
all_game_logs_df = full_df

# Step 3: Filter game logs for the top 5 similar players in their respective season
filtered_game_logs = []

for index, row in top_similar_players.iterrows():
    player_id = row['PLAYER_ID']
    season_year = row['SEASON_YEAR']

    print(f"Filtering game logs for {row['PLAYER_NAME']} in {season_year}...")

    # Filter from all_game_logs_df
    player_logs = all_game_logs_df[
        (all_game_logs_df['PLAYER_ID'] == player_id) & 
        (all_game_logs_df['SEASON_YEAR'] == season_year)
    ]

    filtered_game_logs.append(player_logs)

# Step 4: Combine all filtered logs into new_df
new_df = pd.concat(filtered_game_logs, ignore_index=True) if filtered_game_logs else pd.DataFrame()

new_df

Fetching game logs for season: 2020-21...
Fetching game logs for season: 2021-22...
Fetching game logs for season: 2022-23...
Fetching game logs for season: 2023-24...
Fetching game logs for season: 2024-25...
Top 5 similar players:
       PLAYER_ID         PLAYER_NAME SEASON_YEAR
861     1627783       Pascal Siakam     2021-22
2116    1630532        Franz Wagner     2023-24
573      203944       Julius Randle     2024-25
664     1626157  Karl-Anthony Towns     2022-23
1494    1629628          RJ Barrett     2023-24
Filtering game logs for Pascal Siakam in 2021-22...
Filtering game logs for Franz Wagner in 2023-24...
Filtering game logs for Julius Randle in 2024-25...
Filtering game logs for Karl-Anthony Towns in 2022-23...
Filtering game logs for RJ Barrett in 2023-24...


,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC
0,2021-22,1627783,Pascal Siakam,Pascal,1610612761,TOR,Toronto Raptors,0022101206,2022-04-08T00:00:00,TOR vs. HOU,...,24672,909,887,3985,186,1,131,369,1,39:33
1,2021-22,1627783,Pascal Siakam,Pascal,1610612761,TOR,Toronto Raptors,0022101197,2022-04-07T00:00:00,TOR vs. PHI,...,5520,276,178,18065,62,1,1,114,1,37:01
2,2021-22,1627783,Pascal Siakam,Pascal,1610612761,TOR,Toronto Raptors,0022101182,2022-04-05T00:00:00,TOR vs. ATL,...,1,1601,599,6246,266,1,131,413,1,40:24
3,2021-22,1627783,Pascal Siakam,Pascal,1610612761,TOR,Toronto Raptors,0022101172,2022-04-03T00:00:00,TOR vs. MIA,...,17825,507,887,15346,1262,2151,131,1129,1,43:08
4,2021-22,1627783,Pascal Siakam,Pascal,1610612761,TOR,Toronto Raptors,0022101151,2022-04-01T00:00:00,TOR @ ORL,...,11965,7362,5719,8480,5882,1,131,5928,1,42:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2023-24,1629628,RJ Barrett,RJ,1610612752,NYK,New York Knicks,0022300151,2023-11-06T00:00:00,NYK vs. LAC,...,6163,4296,1681,3132,5347,2248,138,3395,1,30:43
271,2023-24,1629628,RJ Barrett,RJ,1610612752,NYK,New York Knicks,0022300112,2023-10-31T00:00:00,NYK @ CLE,...,13042,6969,6105,2754,13698,2248,138,10751,1,24:51
272,2023-24,1629628,RJ Barrett,RJ,1610612752,NYK,New York Knicks,0022300090,2023-10-28T00:00:00,NYK @ NOP,...,1,2625,4872,23787,10025,2248,138,8454,1,29:59
273,2023-24,1629628,RJ Barrett,RJ,1610612752,NYK,New York Knicks,0022300079,2023-10-27T00:00:00,NYK @ ATL,...,13042,1530,1681,6978,3804,2248,138,2810,1,33:55


In [105]:
#grab all player game logs for given player id from full df
player_game_logs = full_df[full_df['PLAYER_ID'] == chosen_player_id]
player_game_logs

# append the player's game logs to the new_df
player_stats = pd.concat([new_df, player_game_logs], ignore_index=True)


In [106]:
# Define helper functions and parameters

nba_divisions = {
    "Eastern Conference": {
        "ATLANTIC": ["BOS", "BKN", "NYK", "PHI", "TOR"],
        "CENTRAL": ["CHI", "CLE", "DET", "IND", "MIL"],
        "SOUTHEAST": ["ATL", "CHA", "MIA", "ORL", "WAS"]
    },
    "Western Conference": {
        "NORTHWEST": ["DEN", "MIN", "OKC", "POR", "UTA"],
        "PACIFIC": ["GSW", "LAC", "LAL", "PHX", "SAC"],
        "SOUTHWEST": ["DAL", "HOU", "MEM", "NOP", "SAS"]
    }
}

def extract_teams(matchup):
    """Extracts the home and away teams from a matchup string."""
    if " @ " in matchup:
        return matchup.split(" @ ")
    elif " vs. " in matchup:
        return matchup.split(" vs. ")
    return None, None

def find_division(team_name):
    """Finds the division of a team given its abbreviation."""
    for conference, divisions in nba_divisions.items():
        for division, teams in divisions.items():
            if team_name in teams:
                return division
    return None

def is_interdivisional(player_team, opponent):
    """Checks if a game is interdivisional."""
    player_division = find_division(player_team)
    opponent_division = find_division(opponent)
    if player_division and opponent_division and player_division != opponent_division:
        for conference, divisions in nba_divisions.items():
            if player_team in sum(divisions.values(), []) and opponent in sum(divisions.values(), []):
                return 1
    return 0

def classify_rest_days(days):
    """Categorizes the number of rest days into 3 groups."""
    if days < 2:
        return 'Less than 2 days'
    elif 2 <= days <= 3:
        return '2-3 days'
    else:
        return 'More than 3 days'

def add_home_away_teams(dataframe):
    """Assigns the home and away teams based on the MATCHUP column."""
    def split_matchup(row):
        if "vs." in row['MATCHUP']:
            return pd.Series([row['Player_Team'], row['Opponent']], index=['Home_Team', 'Away_Team'])
        elif "@" in row['MATCHUP']:
            return pd.Series([row['Opponent'], row['Player_Team']], index=['Home_Team', 'Away_Team'])
        return pd.Series([None, None], index=['Home_Team', 'Away_Team'])
    dataframe[['Home_Team', 'Away_Team']] = dataframe.apply(split_matchup, axis=1)
    return dataframe

# List of columns for per-minute stats calculations
columns_to_per_minute = [
    'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS'
]

In [107]:
show(player_stats)

In [108]:
import pandas as pd

# Convert GAME_DATE to datetime and filter for games after 2020-01-01
player_stats['GAME_DATE'] = pd.to_datetime(player_stats['GAME_DATE'])
player_stats = player_stats[player_stats['GAME_DATE'] > '2020-01-01']

# Extract player team and opponent from MATCHUP
player_stats['Player_Team'], player_stats['Opponent'] = zip(*player_stats['MATCHUP'].apply(extract_teams))

# Add home court advantage column
player_stats['Home_Court_Advantage'] = player_stats['MATCHUP'].apply(lambda x: 1 if 'vs.' in x else 0)

# Add interdivisional game indicator
player_stats['Interdivisional_Game'] = player_stats.apply(lambda x: is_interdivisional(x['Player_Team'], x['Opponent']), axis=1)

# Sort for rolling calculations at PLAYER_ID level
player_stats = player_stats.sort_values(by=['SEASON_YEAR', 'PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)

# Identify home and away teams
player_stats = add_home_away_teams(player_stats)

# Calculate rest days and categorize them
player_stats['Rest_Days'] = player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])['GAME_DATE'].diff().dt.days
player_stats['Rest_Days'] = player_stats['Rest_Days'].fillna(0).astype(int)
player_stats['Rest_Category'] = player_stats['Rest_Days'].apply(classify_rest_days)

# One-hot encode rest categories
rest_category_dummies = pd.get_dummies(player_stats['Rest_Category'], prefix='Rest_Category')
player_stats = pd.concat([player_stats, rest_category_dummies], axis=1)

# Set target variable and drop any rows with Target == 0
player_stats["Target"] = player_stats['PTS']
player_stats = player_stats.loc[player_stats["Target"] != 0].dropna(subset=["Target"])

# Ensure per-minute stats are calculated at a PLAYER_ID level
columns_to_per_minute = ['PTS', 'AST', 'REB', 'TOV', 'STL', 'BLK', 'FGA', 'FGM', 'FG3A', 'FG3M', 'FTA', 'FTM']

# Compute per-minute stats
for col in columns_to_per_minute:
    per_min_col_name = f"{col}_per_min"
    player_stats[per_min_col_name] = player_stats[col] / player_stats['MIN'].replace(0, 1)  # Avoid division by zero

# Compute rolling averages for per-minute metrics over 3, 7, and 15 game windows
rolling_windows = [3, 7, 15]

for col in columns_to_per_minute:
    per_min_col_name = f"{col}_per_min"
    for window in rolling_windows:
        rolling_col_name = f"{per_min_col_name}_rolling_{window}"
        player_stats[rolling_col_name] = (
            player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'], group_keys=False)[per_min_col_name]
            .rolling(window, min_periods=1)
            .mean()
            .reset_index(level=['SEASON_YEAR', 'PLAYER_ID'], drop=True)  # Reset both levels for index alignment
        )

# Fill any remaining NaN values created by rolling calculations
player_stats.fillna(0, inplace=True)



In [109]:
# Calculate delta metrics for rolling per-minute scoring averages at PLAYER_ID level
player_stats["delta_3_game_avg"] = (
    player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_3"].diff()
)
player_stats["delta_7_game_avg"] = (
    player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_7"].diff()
)
player_stats["delta_15_game_avg"] = (
    player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["PTS_per_min_rolling_15"].diff()
)

# Calculate Usage Rate per minute (approximation)
player_stats["Usage_per_min"] = (
    (player_stats["FGA"] + 0.44 * player_stats["FTA"] + player_stats["TOV"])
    / player_stats["MIN"].replace(0, 1)  # Prevent division by zero
)

# Compute rolling averages for Usage Rate at PLAYER_ID level
for window in [3, 7, 15]:
    rolling_col_name = f"Usage_Rate_rolling_{window}"
    player_stats[rolling_col_name] = (
        player_stats.groupby(['SEASON_YEAR', 'PLAYER_ID'])["Usage_per_min"]
        .rolling(window, min_periods=1)
        .mean()
        .reset_index(level=['SEASON_YEAR', 'PLAYER_ID'], drop=True)  # Align index
    )

# Calculate True Shooting Percentage (TS%)
player_stats["true_shooting_percentage"] = (
    player_stats["PTS"] / (2 * (player_stats["FGA"] + 0.44 * player_stats["FTA"]).replace(0, 1))  # Avoid division by zero
)

# Fill any NaNs that may arise after rolling calculations
player_stats.fillna(0, inplace=True)

In [110]:
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguedashptdefend

# Define Seasons (2020-21 to 2024-25)
seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

# Fetch defensive stats for all seasons
all_defense_data = []

for season in seasons:
    print(f"Fetching defensive stats for {season}...")
    
    try:
        defense_stats = leaguedashptdefend.LeagueDashPtDefend(
            defense_category='Overall',
            per_mode_simple='PerGame',
            season=season,
            season_type_all_star='Regular Season',
            league_id='00'
        )
        defense_df = defense_stats.get_data_frames()[0]
        defense_df["SEASON_YEAR"] = season  # Add season column
        all_defense_data.append(defense_df)
    except Exception as e:
        print(f"Error fetching data for {season}: {e}")

# Combine all seasons into a single DataFrame
defense_df = pd.concat(all_defense_data, ignore_index=True)


Fetching defensive stats for 2020-21...
Fetching defensive stats for 2021-22...
Fetching defensive stats for 2022-23...
Fetching defensive stats for 2023-24...
Fetching defensive stats for 2024-25...


In [111]:
import numpy as np
import pandas as pd

# Rename columns efficiently
player_stats.rename(columns={"PLAYER_ID": "Player_ID", "SEASON_YEAR": "SEASON_YEAR"}, inplace=True)

# Extract primary and secondary positions efficiently
defense_df[["position_1", "position_2"]] = defense_df["PLAYER_POSITION"].str.split("-", expand=True)

# Map player IDs
defense_df["Player_ID"] = defense_df["CLOSE_DEF_PERSON_ID"]

# Create player position DataFrame and merge in one step
player_stats = player_stats.merge(
    defense_df[["Player_ID", "position_1"]].drop_duplicates(), on="Player_ID", how="left"
)

# Compute position-specific averages for defensive FG% (by season)
position_stats = defense_df.groupby(["SEASON_YEAR", "position_1"])["D_FG_PCT"].agg(["mean", "std"])
position_stats.rename(columns={"mean": "pos_avg_fg_pct", "std": "pos_std_fg_pct"}, inplace=True)

# Merge aggregated defensive stats into defense_df
defense_df = defense_df.merge(position_stats, on=["SEASON_YEAR", "position_1"], how="left")

# Calculate the Normalized Defensive Impact Metric (NDIM)
defense_df["NDIM"] = (
    ((defense_df["D_FG_PCT"] - defense_df["pos_avg_fg_pct"]) / defense_df["pos_std_fg_pct"])
    * np.sqrt(defense_df["D_FGA"]) * defense_df["FREQ"]
    + defense_df["PCT_PLUSMINUS"] * (defense_df["GP"] / defense_df["GP"].max())
)

# Sort players by NDIM (lower values indicate better defense)
defense_df.sort_values(by=["SEASON_YEAR", "NDIM"], inplace=True)
defense_df["D_FG_PCT"] = defense_df["D_FG_PCT"].round(3)

# Group by team and position to calculate team-level defensive metrics
grouped_sorted = (
    defense_df.groupby(["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"])["NDIM"]
    .sum()
    .reset_index()
    .sort_values(by=["SEASON_YEAR", "position_1", "NDIM"], ascending=True)
)

# Rank teams by defensive NDIM within each position
grouped_sorted["Def_Rank"] = grouped_sorted.groupby(["SEASON_YEAR", "position_1"])["NDIM"].rank(method="dense", ascending=False)

# Compute team-wide average FG% and variance efficiently
team_defense_stats = (
    defense_df.groupby(["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"])["D_FG_PCT"]
    .agg(["mean", "var"])
    .rename(columns={"mean": "def_FG_PCT", "var": "def_FG_PCT_var"})
    .reset_index()
)

# Merge team-level defensive stats
grouped_sorted = grouped_sorted.merge(team_defense_stats, on=["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "position_1"], how="left")
grouped_sorted.rename(columns={"position_1": "team_position"}, inplace=True)

# Merge defensive summary into player_stats efficiently
player_stats = player_stats.merge(
    grouped_sorted,
    left_on=["SEASON_YEAR", "Opponent", "position_1"],
    right_on=["SEASON_YEAR", "PLAYER_LAST_TEAM_ABBREVIATION", "team_position"],
    how="left"
)


In [112]:
# Read in NBA betting data (adjust the file path if necessary)
betting_data = pd.read_csv("/Users/joshuascantlebury/.cache/kagglehub/datasets/cviaxmiwnptr/nba-betting-data-october-2007-to-june-2024/versions/1/nba_2008-2024.csv")

# Adjust short team abbreviations
nba_3abv = {'GS': 'GSW', 'SA': 'SAS', 'NO': 'NOP', 'NY': 'NYK'}
betting_data["home"] = betting_data["home"].str.upper().replace(nba_3abv)
betting_data["away"] = betting_data["away"].str.upper().replace(nba_3abv)
betting_data["date"] = pd.to_datetime(betting_data["date"])
betting_data["spread"] = betting_data["spread"].replace("0", ".01").astype(float)

# Merge betting data with our main DataFrame
merged_data_2 = player_stats.merge(
    betting_data,
    left_on=["GAME_DATE", "Away_Team", "Home_Team"],
    right_on=["date", "away", "home"],
    how="inner"
)
merged_data_2["delta_fg_pct"] = merged_data_2["def_FG_PCT"] - merged_data_2["FG_PCT"]


In [113]:
show(merged_data_2)

In [114]:
# Define selected features for modeling and drop rows with missing values
selected_features = [
    # Contextual Features
    'Home_Court_Advantage', 'Interdivisional_Game', 
    'Rest_Category_2-3 days',
    'Rest_Category_Less than 2 days',
    'Rest_Category_More than 3 days',
    'spread', 'total',
    
    # Usage Rate and Rolling Averages
    'Usage_Rate_rolling_3', 'Usage_Rate_rolling_7', 'Usage_Rate_rolling_15',
    
    # Rolling Averages (3, 7, 15 games)
    'FGM_per_min_rolling_3', 'FGM_per_min_rolling_7', 'FGM_per_min_rolling_15',
    'REB_per_min_rolling_3', 'REB_per_min_rolling_7', 'REB_per_min_rolling_15',
    'AST_per_min_rolling_3', 'AST_per_min_rolling_7', 'AST_per_min_rolling_15',
    'PTS_per_min_rolling_3', 'PTS_per_min_rolling_7', 'PTS_per_min_rolling_15',
    
    # Delta Metrics
    'delta_3_game_avg', 'delta_7_game_avg', 'delta_15_game_avg',
    
    # Defensive and Advanced Metrics
    'NDIM', 'Def_Rank', 'def_FG_PCT', 'def_FG_PCT_var',
]

In [88]:
merged_data = merged_data_2.dropna(subset=selected_features)

X = merged_data[selected_features]
Y = merged_data["PTS"]

In [89]:
#Make a X + Y data frame
X_Y = pd.concat([X, Y], axis=1)
X_Y

,Home_Court_Advantage,Interdivisional_Game,Rest_Category_2-3 days,Rest_Category_Less than 2 days,Rest_Category_More than 3 days,spread,total,Usage_Rate_rolling_3,Usage_Rate_rolling_7,Usage_Rate_rolling_15,...,PTS_per_min_rolling_7,PTS_per_min_rolling_15,delta_3_game_avg,delta_7_game_avg,delta_15_game_avg,NDIM,Def_Rank,def_FG_PCT,def_FG_PCT_var,PTS
0,1,0,False,True,False,3.5,228.5,0.565618,0.565618,0.565618,...,0.517019,0.517019,0.000000,0.000000,0.000000,5.266036,5.0,0.498714,0.009405,20
1,0,0,True,False,False,2.5,228.5,0.547255,0.547255,0.547255,...,0.444412,0.444412,-0.072607,-0.072607,-0.072607,-3.626922,23.0,0.449000,0.001712,16
2,0,0,True,False,False,1.5,219.5,0.604484,0.604484,0.604484,...,0.487992,0.487992,0.043581,0.043581,0.043581,-4.058004,25.0,0.435800,0.003256,20
3,0,0,False,False,True,1.0,215.5,0.603007,0.593660,0.593660,...,0.467072,0.467072,-0.037569,-0.020920,-0.020920,5.266036,5.0,0.498714,0.009405,10
4,1,0,True,False,False,6.0,217.0,0.619729,0.590739,0.590739,...,0.507194,0.507194,0.098624,0.040121,0.040121,-4.331492,26.0,0.441600,0.001607,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,0,0,True,False,False,4.5,207.5,0.495908,0.481639,0.518265,...,0.539245,0.525554,0.118992,0.037352,0.018125,-9.133287,29.0,0.457636,0.010366,24
465,0,0,True,False,False,12.5,206.5,0.600632,0.512807,0.532148,...,0.556499,0.559401,0.106204,0.017254,0.033847,2.448587,13.0,0.477500,0.000604,22
466,1,1,True,False,False,6.5,211.5,0.650987,0.539310,0.539399,...,0.597422,0.564806,0.062155,0.040924,0.005405,-18.058705,30.0,0.407125,0.012261,16
467,0,1,False,False,True,6.5,215.5,0.643821,0.551846,0.539710,...,0.629952,0.584856,-0.014478,0.032530,0.020049,-4.955541,23.0,0.452857,0.002246,24


In [90]:
# Import additional modeling and evaluation functions
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.feature_selection import f_regression

# %% 
# Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=23)

In [91]:
# Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, Y_train)
lr_predictions = lr_model.predict(X_test)
lr_mape = mean_absolute_percentage_error(Y_test, lr_predictions)

In [92]:
# Ridge Regression Model
ridge_model = Ridge()
ridge_model.fit(X_train, Y_train)
ridge_predictions = ridge_model.predict(X_test)
ridge_mape = mean_absolute_percentage_error(Y_test, ridge_predictions)

In [93]:
# Lasso Regression Model
lasso_model = Lasso()
lasso_model.fit(X_train, Y_train)
lasso_predictions = lasso_model.predict(X_test)
lasso_mape = mean_absolute_percentage_error(Y_test, lasso_predictions)

In [94]:
# Random Forest Regression Model
rf_model = RandomForestRegressor(random_state=23)
rf_model.fit(X_train, Y_train)
rf_predictions = rf_model.predict(X_test)
rf_mape = mean_absolute_percentage_error(Y_test, rf_predictions)


In [95]:
# XGBoost Regression Model
xgb_model = XGBRegressor(random_state=23)
xgb_model.fit(X_train, Y_train)
xgb_predictions = xgb_model.predict(X_test)
xgb_mape = mean_absolute_percentage_error(Y_test, xgb_predictions)

In [96]:
# LightGBM Regression Model
lgb_model = LGBMRegressor(random_state=23)
lgb_model.fit(X_train, Y_train)
lgb_predictions = lgb_model.predict(X_test)
lgb_mape = mean_absolute_percentage_error(Y_test, lgb_predictions)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2396
[LightGBM] [Info] Number of data points in the train set: 343, number of used features: 29
[LightGBM] [Info] Start training from score 21.670554
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [97]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the deep learning model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # Output layer for regression
])

# Compile the model with MAPE metric
model.compile(optimizer='adam', loss='mse', metrics=['mape'])

# Train the model
model.fit(X_train, Y_train, epochs=100, batch_size=16, verbose=1)

xgb_model.fit(X_train, Y_train)
dl_model_predictions = model.predict(X_test)
dl_mape = mean_absolute_percentage_error(Y_test, dl_model_predictions)

Epoch 1/100


/Users/joshuascantlebury/WeekendProjects/Betting App/nba-app/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 626.3862 - mape: 96.8664     
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 55.4132 - mape: 31.8707 
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 56.5974 - mape: 33.5015 
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 48.8559 - mape: 32.1439 
Epoch 5/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 993us/step - loss: 51.7431 - mape: 33.8327
Epoch 6/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 979us/step - loss: 57.3494 - mape: 34.2019
Epoch 7/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 971us/step - loss: 53.0240 - mape: 31.0042
Epoch 8/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - loss: 49.2855 - mape: 32.2585
Epoch 9/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - loss: 49.4855 - mape: 31.9137
Epoch 10/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 955us/step - loss: 46.3803 - mape: 30.3765
Epoch 11/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 56.0718 - mape: 32.7134 
Epoch 12/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 51.0907 - mape:

In [98]:
# Ensemble Model (Weighted Average of Predictions)
mapes = np.array([lr_mape, ridge_mape, lasso_mape, rf_mape, xgb_mape, lgb_mape, dl_mape])
weights = 1 / mapes  # Inverse of MAPE to prioritize better models
weights /= weights.sum()  # Normalize to sum to 1

# Compute ensemble predictions (weighted sum of individual model predictions)
ensemble_predictions = (
    weights[0] * lr_predictions +
    weights[1] * ridge_predictions +
    weights[2] * lasso_predictions +
    weights[3] * rf_predictions +
    weights[4] * xgb_predictions +
    weights[5] * lgb_predictions +
    weights[6] * dl_model_predictions.squeeze()
)

# Evaluate the ensemble model
ensemble_mape = mean_absolute_percentage_error(Y_test, ensemble_predictions)



In [99]:
comparison_df = pd.DataFrame({
    "Actual": Y_test,
    "Linear_Regression": lr_predictions,
    "Ridge": ridge_predictions,
    "Lasso": lasso_predictions,
    "Random_Forest": rf_predictions,
    "XGBoost": xgb_predictions,
    "LightGBM": lgb_predictions,
    "Deep_Learning": dl_model_predictions.squeeze(),
    "Ensemble": ensemble_predictions

})

# Calculate errors
comparison_df["Error_LR"] = abs(comparison_df["Actual"] - comparison_df["Linear_Regression"])
comparison_df["Error_Lasso"] = abs(comparison_df["Actual"] - comparison_df["Lasso"])
comparison_df["Error_Ridge"] = abs(comparison_df["Actual"] - comparison_df["Ridge"])
comparison_df["Error_RF"] = abs(comparison_df["Actual"] - comparison_df["Random_Forest"])
comparison_df["Error_XGB"] = abs(comparison_df["Actual"] - comparison_df["XGBoost"])
comparison_df["Error_LGB"] = abs(comparison_df["Actual"] - comparison_df["LightGBM"])
comparison_df["Error_DL"] = abs(comparison_df["Actual"] - comparison_df["Deep_Learning"])
comparison_df["Error_Ensemble"] = abs(comparison_df["Actual"] - comparison_df["Ensemble"])

summary_df = pd.DataFrame({
    "Model": ["Linear Regression","Ridge","Lasso", "Random Forest", "XGBoost", "LightGBM", "Deep Learning", "Ensemble"],
    "MAPE": [lr_mape,ridge_mape,lasso_mape, rf_mape, xgb_mape, lgb_mape, dl_mape, ensemble_mape]
}).sort_values(by="MAPE")

show(summary_df)
show(comparison_df)



In [124]:
# filter on player id from player stats
player_stats = player_stats[player_stats['Player_ID'] == chosen_player_id]

# Define selected features for modeling and drop rows with missing values
player_stats

,SEASON_YEAR,Player_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,position_1,PLAYER_LAST_TEAM_ABBREVIATION,team_position,NDIM,Def_Rank,def_FG_PCT,def_FG_PCT_var,spread,total,Predicted_PTS
602,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400771,2025-02-12,IND @ WAS,...,F,WAS,F,5.908538,4.0,0.495429,0.002400,0,0,4.043975
601,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400765,2025-02-11,IND vs. NYK,...,F,NYK,F,-2.820265,23.0,0.420600,0.011016,0,0,23.750954
600,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400742,2025-02-08,IND @ LAL,...,F,LAL,F,2.508399,10.0,0.467200,0.001674,0,0,21.942203
599,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400733,2025-02-06,IND @ LAC,...,F,LAC,F,-3.463732,26.0,0.429500,0.007448,0,0,29.676101
598,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400716,2025-02-04,IND @ POR,...,F,POR,F,5.237784,6.0,0.495714,0.004788,0,0,10.511143
597,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400707,2025-02-03,IND @ UTA,...,F,UTA,F,1.315733,13.0,0.466000,0.000799,0,0,21.628784
596,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400686,2025-02-01,IND vs. ATL,...,F,ATL,F,5.239496,5.0,0.484714,0.002449,0,0,22.573073
595,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400664,2025-01-29,IND vs. DET,...,F,DET,F,-0.781609,19.0,0.452375,0.007332,0,0,32.504741
594,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400633,2025-01-25,IND @ SAS,...,F,SAS,F,-0.772931,18.0,0.466778,0.064224,0,0,21.470768
593,2024-25,1627783,Pascal Siakam,Pascal,1610612754,IND,Indiana Pacers,0022400621,2025-01-23,IND @ SAS,...,F,SAS,F,-0.772931,18.0,0.466778,0.064224,0,0,19.155675
